In [2]:
import sys

import torch as t
from agent import Dimensions, Agent
from orchestrator import TarMAC


dims = Dimensions(
  state_dim=18,
  n_actions=5,
  key_dim=16,       # d_k — signature/query size
  value_dim=32,     # d_v — message value size
  gru_input_dim=32, # must equal value_dim for the additive fusion
  gru_hidden_dim=128,  # GRU hidden size
)
n_agents = 3

In [6]:
from torchinfo import summary

model = TarMAC(n_agents, dims)

summary(model)

Layer (type:depth-idx)                   Param #
TarMAC                                   --
├─Agent: 1-1                             --
│    └─Linear: 2-1                       608
│    └─GRU: 2-2                          62,208
│    └─Linear: 2-3                       2,064
│    └─Linear: 2-4                       2,064
│    └─Linear: 2-5                       4,128
│    └─Linear: 2-6                       645
Total params: 71,717
Trainable params: 71,717
Non-trainable params: 0

In [ ]:
import torch as t
from torch.distributions import Categorical

from mpe2 import simple_spread_v3

from agent import Dimensions
from orchestrator import TarMAC

In [ ]:
env = simple_spread_v3.parallel_env(N=3, max_cycles=25, continuous_actions=False)
observations, infos = env.reset(seed=0)

# Fixed agent ordering: the single source of truth for stacking obs and
# unstacking actions. Agent i must mean the same agent in both directions.
agent_names = list(env.agents)

obs_dim = env.observation_space(agent_names[0]).shape[0]
n_actions = env.action_space(agent_names[0]).n
n_agents = len(agent_names)

In [ ]:
dims = Dimensions(
  state_dim=obs_dim,
  n_actions=n_actions,
  key_dim=16,
  value_dim=32,
  gru_input_dim=32,
  gru_hidden_dim=128,
)
model = TarMAC(n_agents, dims)

In [ ]:
# Cell 4 — adapters between the env's dicts and the model's tensors
def stack_obs(obs: dict, names: list[str]) -> t.Tensor:
  return t.stack([t.as_tensor(obs[name], dtype=t.float32) for name in names])

def unstack_actions(actions: t.Tensor, names: list[str]) -> dict:
  return {name: int(actions[i]) for i, name in enumerate(names)}

In [ ]:
def select_actions(logits: t.Tensor) -> tuple[t.Tensor, t.Tensor]:
  dist = Categorical(logits=logits)
  actions = dist.sample()
  return actions, dist.log_prob(actions)

In [ ]:
observations, infos = env.reset(seed=0)
c, h = model.initial_state()

total_reward = 0.0
steps = 0
while env.agents:
  obs = stack_obs(observations, agent_names)

  logits, c, h = model(obs, c, h)
  actions, log_probs = select_actions(logits)

  observations, rewards, terminations, truncations, infos = env.step(
    unstack_actions(actions, agent_names)
  )

  total_reward += sum(rewards.values())
  steps += 1

env.close()
print(f"{steps} steps, total reward {total_reward:.2f}")